# MIPSアセンブラ

## シンタックス
- 1行1命令。
- ラベル無し行：空白＋オペコード＋空白＋オペランド
- ラベル付き行：ラベル＋":"＋空白＋オペコード＋空白＋オペランド
- ラベル：任意の英数字ASCII文字列、大文字小文字は区別される。
- 空白：1個以上の（タブまたはスペース）。
- オペコード内に空白は禁止（当然！）。
- オペランド内に空白を挿入しても良い。

## 疑似コード
- START *addr*
  - これ以降の命令を配置するアドレスを指定する。
- DATA *d1*,*d2*,...
  - 1個以上の32ビットデータをメモリにセットする。
  - *d1*,*d2*,...：32ビットデータ(16進数表現)
- BREAK
  - 実行を停止する。
  - 実際には、opcode=0, funct=C, rs=rt=rd=shamt=0のR形式命令を生成する。

In [8]:
ASMFILE = 'SampleCode.txt'

## データ

In [9]:
Opcode = {'add':0, 'addi':8, 'addiu':9, 'addu':0, 'and':0, 'andi':0xc, 'beq':4,
          'bne':5, 'j':2, 'jal':3, 'jr':0, 'lbu':0x24, 'lhu':0x25, 'll':0x30,
          'lui':0xf, 'lw':0x23, 'nor':0, 'or':0, 'ori':0xd, 'slt':0, 'slti':0xa,
          'sltiu':0xb, 'sltu':0, 'sll':0, 'srl':0, 'sb':0x28, 'sc':0x38,
          'sh':0x29, 'sw':0x2b, 'sub':0, 'subu':0, 'bc1t':0x11, 'bc1f':0x11,
          'div':0, 'divu':0, 'add.s':0x11, 'add.d':0x11, 'c.eq.s':0x11,
          'c.lt.s':0x11, 'c.le.s':0x11, 'c.eq.d':0x11, 'c.lt.d':0x11,
          'c.le.d':0x11, 'div.s':0x11, 'div.d':0x11, 'mul.s':0x11, 'mul.d':0x11,
          'sub.s':0x11, 'sub.d':0x11, 'lwc1':0x31, 'ldc1':0x35, 'mfhi':0,
          'mflo':0, 'mfc0':0x10, 'mult':0, 'multu':0, 'sra':0, 'swc1':0x39,
          'sdc1':0x3d}
Funct = {'add':0x20, 'addu':0x21, 'and':0x24, 'jr':8, 'nor':0x27, 'or':0x25,
         'slt':0x2a, 'sltu':0x2b, 'sll':0, 'srl':2, 'sub':0x22, 'subu':0x23,
         'div':0x1a, 'divu':0x1b, 'add.s':0, 'add.d':0, 'c.eq.s':0x32,
         'c.lt.s':0x3c, 'c.le.s':0x3e, 'c.eq.d':0x32, 'c.lt.d':0x3c,
         'c.le.d':0x3e, 'div.s':3, 'div.d':3, 'mul.s':2, 'mul.d':2, 'sub.s':1,
         'sub.d':1, 'mfhi':0x10, 'mflo':0x12, 'mfc0':0, 'mult':0x18,
         'multu':0x19, 'sra':3}
Fmt = {'bclt':8, 'bclf':8, 'add.s':0x10, 'add.d':0x11, 'c.eq.s':0x10,
       'c.lt.s':0x10, 'c.le.s':0x10, 'c.eq.d':0x11, 'c.lt.d':0x11,
       'c.le.d':0x11, 'div.s':0x10, 'div.d':0x11, 'mul.s':0x10, 'mul.d':0x11,
       'sub.s':0x10, 'sub.d':0x11}
Ft = {'bc1t':1, 'bc1f':0}
Register = {'$zero':0, '$at':1, '$v0':2, '$v1':3, '$a0':4, '$a1':5, '$a2':6,
            '$a3':7, '$t0':8, '$t1':9, '$t2':10, '$t3':11, '$t4':12, '$t5':13,
            '$t6':14, '$t7':15, '$s0':16, '$s1':17, '$s2':18, '$s3':19,
            '$s4':20, '$s5':21, '$s6':22, '$s7':23, '$t8':24, '$t9':26,
            '$k0':26, '$k1':27, '$gp':28, '$sp':29, '$fp':30, '$ra':31}
Register2 = {'$f0':0, '$f1':1, '$f2':2, '$f3':3, '$f4':4, '$f5':5, '$f6':6,
             '$f7':7, '$f8':8, '$f9':9, '$f10':10, '$f11':11, '$f12':12,
             '$f13':13, '$f14':14, '$f15':15, '$f16':16, '$f17':17, '$f18':18,
             '$f19':19, '$f20':20, '$f21':21, '$f22':22, '$f23':23, '$f24':24,
             '$f25':25, '$f26':26, '$f27':27, '$f28':28, '$f29':29, '$f30':30,
             '$f31':31}
Register3 = {'$'+str(i):i for i in range(0,32)}
Operand = {'add':'rd,rs,rt', 'addi':'rt,rs,imm', 'addiu':'rt,rs,imm',
           'addu':'rd,rs,rt', 'and':'rd,rs,rt', 'andi':'rt,rs,imm',
           'beq':'rt,rs,addr', 'bne':'rt,rs,addr', 'j':'addr', 'jal':'addr',
           'jr':'rs', 'lbu':'rt,imm(rs)', 'lhu':'rt,imm(rs)', 'll':'rt,imm(rs)',
           'lui':'rt,imm', 'lw':'rt,imm(rs)', 'nor':'rd,rs,rt', 'or':'rd,rs,rt',
           'ori':'rt,rs,imm', 'slt':'rd,rs,rt', 'slti':'rt,rs,imm',
           'sltiu':'rt,rs,imm', 'sltu':'rd,rs,rt', 'sll':'rd,rt,shamt',
           'srl':'rd,rt,shamt', 'sb':'rt,imm(rs)', 'sc':'rt,imm(rs)',
           'sh':'rt,imm(rs)', 'sw':'rt,imm(rs)', 'sub':'rd,rs,rt',
           'subu':'rd,rs,rt', 'bc1t':'imm', 'bc1f':'imm', 'div':'rs,rt',
           'divu':'rs,rt', 'add.s':'fd,fs,ft', 'add.d':'fd,fs,ft',
           'c.eq.s':'fs,ft', 'c.lt.s':'fs,ft', 'c.le.s':'fs,ft',
           'c.eq.d':'fs,ft', 'c.lt.d':'fs,ft', 'c.le.d':'fs,ft',
           'div.s':'fd,fs,ft', 'div.d':'fd,fs,ft', 'mul.s':'fd,fs,ft',
           'mul.d':'fd,fs,ft', 'sub.s':'fd,fs,ft', 'sub.d':'fd,fs,ft',
           'lwc1':'ft,imm(rs)', 'ldc1':'ft,imm(rs)', 'mfhi':'rd', 'mflo':'rd',
           'mfc0':'rd,rs', 'mult':'rs,rt', 'multu':'rs,rt', 'sra':'rd,rt,shamt',
           'swc1':'ft,imm(rs)', 'sdc1':'ft,imm(rs)'}

## 読み込み

In [10]:
with open(ASMFILE, "r") as f :
  lines = f.read().splitlines()
lines

['    START   0',
 '    add     $s0,$zero,$zero',
 '    addi    $s1,$zero,1',
 '    addi    $s2,$zero,11',
 'LOOP: beq   $s1,$s2,END',
 '    add     $s0,$s0,$s1',
 '    addi    $s1,$s1,1',
 '    j       LOOP',
 'END: BREAK']

## パス１：ラベル計算

In [11]:
Lines = list()
Labels = dict()
pc = 0
for l in lines :
  if len(l) == 0 :
    continue
  x = l.split();
  if l[0].isalnum() :
    label = x.pop(0)
    if label[-1] != ':' :
      print('label error:', l);
      continue;
    label = label[:-1]
    if not label.isalnum() :
      print('label error:', l);
    Labels[label] = pc + 0
  opcode = x.pop(0)
  operand = ''.join(x)
  Lines.append([opcode, operand])
  if opcode == 'START' :
    pc = int(operand)
  elif opcode == 'DATA' :
    pc += len(operand.split()) * 4
  else :
    pc += 4
Labels

{'LOOP': 12, 'END': 28}

## パス２：コード生成

In [12]:
def typeR(op,rs,rt,rd,shmat,funct) :
  return (op<<26)+(rs<<21)+(rt<<16)+(rd<<11)+(shmat<<6)+funct
def typeI(op,rs,rt,imm) :
  return (op<<26)+(rs<<21)+(rt<<16)+imm
def typeJ(op,addr) :
  return (op<<26)+addr
def typeFR(op,fmt,ft,fs,fd,funct) :
  return (op<<26)+(fmt<<21)+(ft<<16)+(fs<<11)+(fd<<6)+funct
def typeFI(op,fmt,ft,imm) :
  return (op<<26)+(fmt<<21)+(ft<<16)+imm
def evalConst(imm) :
  if imm in Labels.keys() :
    imm = Labels[imm]
  else :
    imm = int(imm)
  return imm

In [13]:
Codes = list()
PC = 0
for l in Lines :
#  print(l) # DEBUG
  (opcode, operand) = (l[0], l[1])
  match opcode :
    case 'START' :
      PC = int(operand)
      continue
    case 'BREAK' :
      code = 0x0000000c
    case 'DATA' :
      for data in operand.split(',') :
        Codes.append(int(data))
        pc += 4
      continue
    case x if Operand[x] == "rd,rs,rt" :
      op = Opcode[opcode]
      f = Funct[opcode]
      (rd,rs,rt) = [Register[reg] for reg in operand.split(',')]
      code = typeR(op,rs,rt,rd,0,f)
    case x if Operand[x] == "rd,rt,shamt" :
      op = Opcode[opcode]
      f = Funct[opcode]
      (rd,rt,shamt) = operand.split(',')
      rd = Register[rd]
      rt = Register[rt]
      shamt = int(shamt) & 0x1f
      code = typeR(op,0,rt,rd,shamt,f)
    case x if Operand[x] == "rd,rs" :
      op = Opcode[opcode]
      f = Funct[opcode]
      (rd,rs) = operand.split(',')
      rd = Register[rd]
      rs = Register3[rs]
      code = typeR(op,rs,0,rd,0,f)
    case x if Operand[x] == "rs,rt" :
      op = Opcode[opcode]
      f = Funct[opcode]
      (rs,rt) = [Register[reg] for reg in operand.split(',')]
      code = typeR(op,rs,rt,0,0,f)
    case x if Operand[x] == "rd" :
      op = Opcode[opcode]
      f = Funct[opcode]
      rd = Register[operand]
      code = typeR(op,0,rd,0,0,f)
    case x if Operand[x] == "rs" :
      op = Opcode[opcode]
      f = Funct[opcode]
      rs = Register[operand]
      code = typeR(op,rs,0,0,0,f)
    case x if Operand[x] == "rt,rs,imm" :
      op = Opcode[opcode]
      (rt,rs,imm) = operand.split(',')
      rt = Register[rt]
      rs = Register[rs]
      imm = evalConst(imm) & 0xffff
      code = typeI(op,rs,rt,imm)
    case x if Operand[x] == "rt,rs,addr" :
      op = Opcode[opcode]
      (rt,rs,addr) = operand.split(',')
      rt = Register[rt]
      rs = Register[rs]
      imm = ((evalConst(addr)-PC-2)//4) & 0xffff
      code = typeI(op,rs,rt,imm)
    case x if Operand[x] == "rt,imm(rs)" :
      op = Opcode[opcode]
      (rt,immrs) = operand.split(',')
      (imm,rs) = immrs.split('(')
      if rs[-1] != ')' :
        print("')' is required:", opcode, operand)
      rs = rs[:-1]
      rt = Register[rt]
      rs = Register[rs]
      imm = evalConst(imm) & 0xffff
      code = typeI(op,rs,rt,imm)
    case x if Operand[x] == "rt,imm" :
      op = Opcode[opcode]
      (rt,imm) = operand.split(',')
      rt = Register[rt]
      imm = evalConst(imm) & 0xffff
      code = typeI(op,0,rt,imm)
    case x if Operand[x] == "ft,imm(rs)" :
      op = Opcode[opcode]
      (ft,immrs) = operand.split(',')
      (imm,rs) = immrs.split('(')
      if rs[-1] != ')' :
        print("')' is required:", opcode, operand)
      rs = rs[:-1]
      rt = Register2[ft]
      rs = Register[rs]
      imm = evalConst(imm) & 0xffff
      code = typeI(op,rs,rt,imm)
    case x if Operand[x] == "addr" :
      op = Opcode[opcode]
      imm = (evalConst(operand)//4) & 0x3fffffff
      code = typeJ(op,imm)
    case x if Operand[x] == "fd,fs,ft" :
      op = Opcode[opcode]
      fmt = Fmt[opcode]
      funct = Funct[opcode]
      (fd,fs,ft) = [Register2[reg] for reg in operand.split(',')]
      code = typeFR(op,fmt,ft,fs,fd,funct)
    case x if Operand[x] == "fs,ft" :
      op = Opcode[opcode]
      fmt = Fmt[opcode]
      funct = Funct[opcode]
      (fs,ft) = [Register2[reg] for reg in operand.split(',')]
      code = typeFR(op,fmt,ft,fs,0,funct)
    case x if Operand[x] == "imm" :
      op = Opcode[opcode]
      fmt = 8
      ft = Ft[opcode]
      imm = ((evalConst(operand)-PC-2)//4) & 0xffff
      code = typeFI(op,fmt,ft,imm)
    case _:
      print(opcode)
      continue
#  print(code) # DEBUG
  Codes.append(code)
  PC += 4
for c in Codes :
  print(format(c,'#034b'))

0b00000000000000001000000000100000
0b00100000000100010000000000000001
0b00100000000100100000000000001011
0b00010010010100010000000000000011
0b00000010000100011000000000100000
0b00100010001100010000000000000001
0b00001000000000000000000000000011
0b00000000000000000000000000001100
